
%md
## Metodología aplicada en el proyecto RNDC

El flujo seguido en Databricks se organizó en capas para transformar los datos de transporte RNDC:



### Etapas del proceso

- **Bronze → Ingesta de datos**  
  Se cargan los 12 archivos de cada uno de los meses de 2024 de la pagina dEL RNDC en bruto dentro de Databricks, se crean las tablas de cada uno de los Dataset dentro del catálogo, lugo se frea un DataFrema anual conserevando su estructura original.  



- **Gold → Análisis y preparación**  
  Se filtraron los camiones de tres ejes, se identificaron las rutas más transitadas y se calcularon tarifas promedio.  
  Estos datos se utilizaron para entrenar un modelo de regresión lineal que proyecta tarifas en función del peso.  

### Nota sobre Web Scraping

Durante el desarrollo del proyecto no fue posible implementar el web scraping, ya que la página oficial del RNDC utiliza un sistema de CAPTCHA que bloquea la automatización de respuestas.  
Por esta razón, se trabajó con el archivo descargado manualmente, manteniendo la lógica del flujo de datos y asegurando la continuidad del análisis.

### Resultado

El modelo mostró un intercepto dominante y un coeficiente de peso bajo, lo que generó predicciones muy similares para distintos valores de peso.  
Este hallazgo evidencia la necesidad de enriquecer el dataset con más variables para mejorar la precisión de las proyecciones.


In [0]:
spark.sql("SHOW TABLES").show()


+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [0]:
print(df_bronze.columns)


['MES', 'COD_CONFIG_VEHICULO', 'CONFIG_VEHICULO', 'CODOPERACIONTRANSPORTE', 'OPERACIONTRANSPORTE', 'CODTIPOCONTENEDOR', 'TIPOCONTENEDOR', 'CODMUNICIPIOORIGEN', 'MUNICIPIOORIGEN', 'DEPARTAMENTOORIGEN', 'CODMUNICIPIODESTINO', 'MUNICIPIODESTINO', 'DEPARTAMENTODESTINO', 'CODMERCANCIA', 'MERCANCIA', 'NATURALEZACARGA', 'VIAJESTOTALES', 'KILOGRAMOS', 'GALONES', 'VIAJESLIQUIDOS', 'VIAJESVALORCERO', 'KILOMETROS', 'VALORESPAGADOS', 'CODMUNICIPIOINTERMEDIO', 'MUNICIPIOINTERMEDIO', 'DEPARTAMENTOINTERMEDIO', 'KILOMETROSREGRESO', 'KILOGRAMOSREGRESO', 'GALONESREGRESO']


In [0]:
df_enero = spark.read.table("workshop.default.estadisticas_rndc_202401")
df_enero.show(5)


+------+-------------------+--------------------+----------------------+-------------------+-----------------+--------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+------------+--------------------+---------------+-------------+----------+-------+--------------+---------------+----------+--------------+----------------------+-------------------+----------------------+-----------------+-----------------+--------------+
|   MES|COD_CONFIG_VEHICULO|     CONFIG_VEHICULO|CODOPERACIONTRANSPORTE|OPERACIONTRANSPORTE|CODTIPOCONTENEDOR|TIPOCONTENEDOR|CODMUNICIPIOORIGEN|   MUNICIPIOORIGEN|DEPARTAMENTOORIGEN|CODMUNICIPIODESTINO|    MUNICIPIODESTINO|DEPARTAMENTODESTINO|CODMERCANCIA|           MERCANCIA|NATURALEZACARGA|VIAJESTOTALES|KILOGRAMOS|GALONES|VIAJESLIQUIDOS|VIAJESVALORCERO|KILOMETROS|VALORESPAGADOS|CODMUNICIPIOINTERMEDIO|MUNICIPIOINTERMEDIO|DEPARTAMENTOINTERMEDIO|KILOMETROSREGRESO|KILOGRAMOSREGRESO|GALONESREGRESO

In [0]:
base_tabla = "workshop.default.estadisticas_rndc_"


In [0]:
meses = [
    "202401","202402","202403","202404",
    "202405","202406","202407","202408",
    "202409","202410","202411","202412"
]

df_bronze = None
for mes in meses:
    tabla = base_tabla + mes
    df_mes = spark.read.table(tabla)
    if df_bronze is None:
        df_bronze = df_mes
    else:
        df_bronze = df_bronze.union(df_mes)

print("Total registros Bronze:", df_bronze.count())
df_bronze.show(5)


Total registros Bronze: 2083061
+------+-------------------+--------------------+----------------------+-------------------+-----------------+--------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+------------+--------------------+---------------+-------------+----------+-------+--------------+---------------+----------+--------------+----------------------+-------------------+----------------------+-----------------+-----------------+--------------+
|   MES|COD_CONFIG_VEHICULO|     CONFIG_VEHICULO|CODOPERACIONTRANSPORTE|OPERACIONTRANSPORTE|CODTIPOCONTENEDOR|TIPOCONTENEDOR|CODMUNICIPIOORIGEN|   MUNICIPIOORIGEN|DEPARTAMENTOORIGEN|CODMUNICIPIODESTINO|    MUNICIPIODESTINO|DEPARTAMENTODESTINO|CODMERCANCIA|           MERCANCIA|NATURALEZACARGA|VIAJESTOTALES|KILOGRAMOS|GALONES|VIAJESLIQUIDOS|VIAJESVALORCERO|KILOMETROS|VALORESPAGADOS|CODMUNICIPIOINTERMEDIO|MUNICIPIOINTERMEDIO|DEPARTAMENTOINTERMEDIO|KILOMETROSREGRESO|


# Conclusión – Limitaciones y Aprendizajes

Durante el desarrollo del proyecto se identificó una limitación importante:  

Desde la página del RNDC únicamente permite la descarga de los datasets mes a mes, lo que obliga a realizar la carga de la información manualmente.  

A pesar de esta restricción, se subieron los archisvos como tablas en Databricks y luego unirlos en un DataFrame anual lo que permitió garantizar que los datos estén disponibles de forma consolidada para análisis posteriores, y deja claro el valor de una arquitectura organizada en capas.  

El aprendizaje principal es que, incluso con limitaciones en la fuente, una buena práctica de carga de dataset y transformación permite construir un flujo sólido y escalable.



# Capa Silver – Limpieza y Transformación

En esta etapa se realizan las transformaciones básicas sobre los datos de la capa Bronze: 
 
- Normalización de nombres de columnas.  
- Conversión de variables numéricas a su tipo correcto.  
- Eliminación de duplicados y registros nulos.  

El objetivo de la capa Silver es obtener un dataset limpio y estructurado, listo para análisis más avanzados.


In [0]:
from pyspark.sql.functions import col

df_silver = df_bronze \
    .withColumn("KILOGRAMOS", col("KILOGRAMOS").cast("double")) \
    .withColumn("KILOMETROS", col("KILOMETROS").cast("double"))

df_silver = df_silver.dropDuplicates()
df_silver = df_silver.dropna(subset=["KILOGRAMOS","KILOMETROS"])

print("Total registros Silver:", df_silver.count())
df_silver.show(5)


Total registros Silver: 2083061
+------+-------------------+--------------------+----------------------+-------------------+-----------------+--------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+------------+--------------------+---------------+-------------+----------+-------+--------------+---------------+----------+--------------+----------------------+-------------------+----------------------+-----------------+-----------------+--------------+
|   MES|COD_CONFIG_VEHICULO|     CONFIG_VEHICULO|CODOPERACIONTRANSPORTE|OPERACIONTRANSPORTE|CODTIPOCONTENEDOR|TIPOCONTENEDOR|CODMUNICIPIOORIGEN|   MUNICIPIOORIGEN|DEPARTAMENTOORIGEN|CODMUNICIPIODESTINO|    MUNICIPIODESTINO|DEPARTAMENTODESTINO|CODMERCANCIA|           MERCANCIA|NATURALEZACARGA|VIAJESTOTALES|KILOGRAMOS|GALONES|VIAJESLIQUIDOS|VIAJESVALORCERO|KILOMETROS|VALORESPAGADOS|CODMUNICIPIOINTERMEDIO|MUNICIPIOINTERMEDIO|DEPARTAMENTOINTERMEDIO|KILOMETROSREGRESO|

In [0]:
from pyspark.sql.functions import col

# Conversión de columnas numéricas
df_silver = df_bronze \
    .withColumn("KILOGRAN", col("KILOGRAN").cast("double")) \
    .withColumn("VALORSPC", col("VALORSPC").cast("double"))

# Eliminar duplicados
df_silver = df_silver.dropDuplicates()

# Eliminar registros con valores nulos en columnas clave
df_silver = df_silver.dropna(subset=["KILOGRAN","VALORSPC"])

print("Total registros Silver:", df_silver.count())
df_silver.show(5)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6543441434643901>, line 14
     11 # Eliminar registros con valores nulos en columnas clave
     12 df_silver = df_silver.dropna(subset=["KILOGRAN","VALORSPC"])
---> 14 print("Total registros Silver:", df_silver.count())
     15 df_silver.show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1929     query = self._plan.to_proto(self._session.client)
-> 1

| Columna | Nombre sugerido | Ejemplo de valor | Significado probable |
| --- | --- | --- | --- |
| ``_c0`` | periodo | 202401 | Año/mes del registro |
| ``_c1`` | codigo | CA | Código de servicio / empresa |
| ``_c2`` | vehiculo | Camioneta de 2 ejes | Tipo de vehículo |
| ``_c3`` | categoria | G | Categoría (General, etc.) |
| ``_c4`` | tipo_servicio | General | Tipo de servicio |
| ``_c5`` | indicador1 | . | Campo auxiliar |
| ``_c6`` | indicador2 | . | Campo auxiliar |
| ``_c7`` | peso | 5001000.0 | Peso transportado |
| ``_c8`` | origen | MEDELLIN ANTIOQUIA | Ciudad/Departamento origen |
| ``_c9`` | depto_origen | ANTIOQUIA | Departamento origen |
| ``_c10`` | valor | 1.9142E7 | Valor monetario |
| ``_c11`` | destino | CALOTO CAUCA | Ciudad destino |
| ``_c12`` | depto_destino | CAUCA | Departamento destino |
| ``_c13`` | codigo_producto | 009980 | Código producto |
| ``_c14`` | producto | PRODUCTOS VARIOS | Tipo de producto |
| ``_c15`` | modalidad | Carga Normal | Modalidad de carga |
| ``_c16`` | cantidad | 1.0 | Cantidad de viajes/unidades |
| ``_c17`` | distancia | 500.0 | Distancia (km) |
| ``_c18`` | peajes | 0.0 | Valor peajes |
| ``_c19`` | otros_costos | 0.0 | Otros costos |
| ``_c20`` | descuento | 0.0 | Descuento aplicado |
| ``_c21`` | tiempo_viaje | 465.0 | Tiempo estimado (minutos) |
| ``_c22`` | tarifa | 750000.0 | Tarifa cobrada |
| ``_c23`` | indicador3 | 0.0 | Campo auxiliar |
| ``_c24`` | indicador4 | . | Campo auxiliar |
| ``_c25`` | indicador5 | . | Campo auxiliar |
| ``_c26`` | indicador6 | 0.0 | Campo auxiliar |
| ``_c27`` | indicador7 | 0.0 | Campo auxiliar |
| ``_c28`` | indicador8 | 0.0 | Campo auxiliar |

In [0]:
df_silver = df_bronze \
    .withColumnRenamed("_c0", "periodo") \
    .withColumnRenamed("_c2", "vehiculo") \
    .withColumnRenamed("_c7", "peso") \
    .withColumnRenamed("_c8", "origen") \
    .withColumnRenamed("_c11", "destino") \
    .withColumnRenamed("_c10", "valor") \
    .withColumnRenamed("_c22", "tarifa")


In [0]:
from pyspark.sql.functions import col

df_silver = df_silver \
    .withColumn("peso", col("peso").cast("double")) \
    .withColumn("valor", col("valor").cast("double")) \
    .withColumn("tarifa", col("tarifa").cast("double")) \
    .withColumn("distancia", col("_c17").cast("double")) \
    .withColumn("tiempo_viaje", col("_c21").cast("double"))


In [0]:
df_silver = df_silver.dropDuplicates()
df_silver = df_silver.dropna(subset=["vehiculo","origen","destino","tarifa"])


In [0]:
df_silver = df_bronze.withColumnRenamed("_c7", "peso")
df_silver = df_silver.withColumn("peso", col("peso").cast("double"))


In [0]:
# Renombrar todas las columnas según tu mapeo
df_silver = df_bronze.toDF(
    "periodo","codigo","vehiculo","categoria","tipo_servicio",
    "indicador1","indicador2","peso","origen","depto_origen",
    "valor","destino","depto_destino","codigo_producto","producto",
    "modalidad","cantidad","distancia","peajes","otros_costos",
    "descuento","tiempo_viaje","tarifa","indicador3","indicador4",
    "indicador5","indicador6","indicador7","indicador8"
)

# Convertir columnas numéricas a double
from pyspark.sql.functions import col

df_silver = df_silver \
    .withColumn("peso", col("peso").cast("double")) \
    .withColumn("valor", col("valor").cast("double")) \
    .withColumn("tarifa", col("tarifa").cast("double")) \
    .withColumn("distancia", col("distancia").cast("double")) \
    .withColumn("tiempo_viaje", col("tiempo_viaje").cast("double"))

# Eliminar duplicados y nulos
df_silver = df_silver.dropDuplicates()
df_silver = df_silver.dropna(subset=["vehiculo","origen","destino","tarifa","peso"])

print("Total registros Silver:", df_silver.count())
df_silver.show(5)


Total registros Silver: 2082967
+-------+------+--------------------+---------+-------------+----------+----------+---------+------------------+------------+---------+--------------------+-------------+---------------+--------------------+------------+--------+---------+------+------------+---------+------------+---------+----------+----------+----------+----------+----------+----------+
|periodo|codigo|            vehiculo|categoria|tipo_servicio|indicador1|indicador2|     peso|            origen|depto_origen|    valor|             destino|depto_destino|codigo_producto|            producto|   modalidad|cantidad|distancia|peajes|otros_costos|descuento|tiempo_viaje|   tarifa|indicador3|indicador4|indicador5|indicador6|indicador7|indicador8|
+-------+------+--------------------+---------+-------------+----------+----------+---------+------------------+------------+---------+--------------------+-------------+---------------+--------------------+------------+--------+---------+------+----

**Conclusión de Capa Silver – Limpieza y Transformación**

En esta etapa se aplican transformaciones sobre los datos crudos de la capa Bronze: 

- Se renombraron las columnas clave  (periodo, vehiculo, peso, origen, destino, valor, tarifa) para mejorar la legibilidad.  
- Se convirtieron a tipo numérico las columnas de medidas y valores (peso, valor, tarifa, distancia, tiempo_viaje).  
- Se eliminaron duplicados y registros nulos en campos esenciales.  

El resultado es un dataset estructurado y limpio, que facilita el análisis en la capa Gold.



# Capa Gold – Análisis y Métricas

En esta etapa se construyen los datasets finales para análisis y visualización.  

A partir de los datos limpios de la capa Silver, se generan indicadores clave como: 
 
- Promedio de tarifas por municipio de origen.  
- Rutas más frecuentes (origen → destino).  
- Distribución de peso transportado por tipo de vehículo.  

La capa Gold representa la información lista para ser consumida en herramientas de BI como Power BI, facilitando la toma de decisiones estratégicas.


In [0]:
from pyspark.sql.functions import col, avg, count

# 1. Promedio de tarifas por municipio de origen
df_tarifas = df_silver.groupBy("origen").agg(avg("tarifa").alias("tarifa_promedio"))
df_tarifas.show(10)

# 2. Rutas más frecuentes (origen → destino)
df_rutas = df_silver.groupBy("origen","destino").agg(count("*").alias("frecuencia")) \
                    .orderBy(col("frecuencia").desc())
df_rutas.show(10)

# 3. Distribución de peso por tipo de vehículo
df_peso_vehiculo = df_silver.groupBy("vehiculo").agg(avg("peso").alias("peso_promedio"))
df_peso_vehiculo.show(10)


+--------------------+--------------------+
|              origen|     tarifa_promedio|
+--------------------+--------------------+
|  MEDELLIN ANTIOQUIA|    5787191.87394112|
|     AMAGA ANTIOQUIA|1.1944513522127487E7|
|ANGELOPOLIS ANTIO...|   7271483.253731343|
|SANTAFE DE ANTIOQ...|   2354362.402745995|
|  APARTADO ANTIOQUIA|   4684454.999599359|
| ARBOLETES ANTIOQUIA|  3923614.6226415094|
|   ARMENIA ANTIOQUIA|   2113387.126213592|
|   BARBOSA ANTIOQUIA|   6465930.338223418|
|EL HATILLO BARBOS...|   4045826.136842105|
|     BELLO ANTIOQUIA|   6130557.971848608|
+--------------------+--------------------+
only showing top 10 rows
+--------------------+--------------------+----------+
|              origen|             destino|frecuencia|
+--------------------+--------------------+----------+
|BUENAVENTURA VALL...| BOGOTA BOGOTA D. C.|     22444|
|   CARTAGENA BOLIVAR| BOGOTA BOGOTA D. C.|     16261|
| BOGOTA BOGOTA D. C.|   CARTAGENA BOLIVAR|      8240|
|BUENAVENTURA VALL...|  MEDEL

In [0]:
# Filtrar únicamente vehículos que contengan "3 ejes"
df_3ejes = df_silver.filter(df_silver["vehiculo"].contains("3 ejes"))


In [0]:
from pyspark.sql.functions import count, avg

# Agrupar por origen-destino y calcular métricas
df_rutas_top10 = df_3ejes.groupBy("origen","destino") \
    .agg(
        count("*").alias("num_viajes"),
        avg("tarifa").alias("tarifa_promedio")
    ) \
    .orderBy(col("num_viajes").desc()) \
    .limit(10)

df_rutas_top10.show(truncate=False)


+----------------------------+-----------------------+----------+--------------------+
|origen                      |destino                |num_viajes|tarifa_promedio     |
+----------------------------+-----------------------+----------+--------------------+
|BUENAVENTURA VALLE DEL CAUCA|BOGOTA BOGOTA D. C.    |11714     |1.8156462984121565E7|
|CARTAGENA BOLIVAR           |BOGOTA BOGOTA D. C.    |7069      |2.005723229084736E7 |
|BUENAVENTURA VALLE DEL CAUCA|YUMBO VALLE DEL CAUCA  |5643      |8935591.309587099   |
|BUENAVENTURA VALLE DEL CAUCA|MEDELLIN ANTIOQUIA     |4537      |1.231378070994049E7 |
|BUENAVENTURA VALLE DEL CAUCA|CALI VALLE DEL CAUCA   |4508      |9089902.727373559   |
|CARTAGENA BOLIVAR           |BARRANQUILLA ATLANTICO |4348      |6685861.633854646   |
|BUENAVENTURA VALLE DEL CAUCA|FUNZA CUNDINAMARCA     |3254      |1.391762947510756E7 |
|CARTAGENA BOLIVAR           |MEDELLIN ANTIOQUIA     |3030      |1.228223105610561E7 |
|BUENAVENTURA VALLE DEL CAUCA|PALMIRA VALLE


# Conclusión – Capa Gold

El análisis de la capa Gold permitió identificar las 10 rutas con mayor tráfico de camiones de 3 ejes en Colombia. 

Entre ellas destacan corredores estratégicos como Buenaventura → Bogotá, Cartagena → Bogotá y Buenaventura → Medellín, que concentran un alto volumen de viajes y tarifas promedio elevadas.  

Estos resultados son fundamentales para un análisis financiero, ya que permiten:  
- Evaluar la rentabilidad de las rutas más transitadas.  
- Identificar corredores viales críticos donde el alto tráfico puede presionar los costos.  
- Priorizar rutas estratégicas para la optimización de tarifas y márgenes.  

La capa Gold consolida la información en métricas claras y accionables con el fin de  apoyar la toma de decisiones en logística y transporte.


In [0]:
df_rutas_top10.toPandas().to_csv("/Volumes/workshop/default/Volumen_rndc/rutas_top10.csv", index=False)
